<a href="https://colab.research.google.com/github/clapg/qualidade_vida_mg/blob/main/analise_qualidade_vida_uberlandia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Qualidade de vida nos municípios de Minas Gerais**
### Análise exploratória de indicadores sociais em 853 municípios de Minas Gerais, com foco em Uberlândia

**Fontes:**
- **IMRS 2020** — Índice Mineiro de Responsabilidade Social | Fundação João Pinheiro (FJP)
- **Atlas Brasil** — IDHM histórico 1991–2010 | PNUD Brasil, IPEA e FJP
- **IBGE** — Censo 2022, PIB per capita 2023, Mortalidade Infantil 2023
- **Atlas Brasil 2021** — Pobreza e analfabetismo | PNUD Brasil, IPEA e FJP


### Perguntas de negócio:

1.	Como a cidade se posiciona entre os 853 municípios de MG?
2.	O IDHM de Uberlândia avançou entre 1991 e 2010? Em quais dimensões o progresso foi mais expressivo?
3.	Os indicadores mais recentes (Censo 2022, PIB 2023, Mortalidade Infantil 2023) estão alinhados com a trajetória de desenvolvimento apontada pelo IDHM histórico?
4.	Uberlândia lidera os indicadores de qualidade de vida entre as maiores cidades de MG, ou há municípios com desempenho superior considerados como referência?
5.	Quais dimensões do IDHM ainda ficam abaixo da média estadual ou nacional e representam lacunas prioritárias para a gestão pública?


In [ ]:
# Conecta o Google Drive ao ambiente do Colab
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Caminho dos arquivos do projeto no Google Drive
PASTA = '/content/drive/MyDrive/Dados/Projetos/qualidade de vida/'

data_xlsx      = PASTA + 'data.xlsx'
atlas_xlsx     = PASTA + 'atlas_brasil_2021.xlsx'
ibge_csv       = PASTA + 'ibge.csv'
dados_consulta = PASTA + 'DadosConsulta.csv'

## 1. Configuração e importações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

In [ ]:
# Aplica configurações visuais padrão a todos os gráficos gerados na sessão.
# Usar rcParams.update() garante consistência sem precisar repetir parâmetros
# em cada chamada de plt.subplots() ou ax individualmente.
plt.rcParams.update({
    'figure.facecolor': 'white',      # fundo externo da figura (área além dos eixos)
    'axes.facecolor':   '#f8f9fa',    # fundo interno dos eixos (área do gráfico)
    'axes.grid':        True,         # exibe grade por padrão em todos os eixos
    'grid.color':       'white',      # cor branca para a grade contrastar com o fundo cinza
    'grid.linewidth':   1.2,          # espessura das linhas da grade
    'font.family':      'DejaVu Sans',# fonte padrão — compatível com acentos do português
    'axes.spines.top':  False,        # remove a borda superior dos eixos
    'axes.spines.right':False,        # remove a borda direita dos eixos
})

# Dicionário centralizado com as cores institucionais/semânticas utilizadas nos gráficos.
# Usar um dicionário nomeado (em vez de códigos hex avulsos) facilita a manutenção:
# qualquer ajuste de cor é feito aqui e reflete automaticamente em todos os gráficos.
CORES = {
    'uberlandia': '#1a6fb5',   # azul — série referente a Uberlândia
    'mg':         '#e07b00',   # laranja — série referente a Minas Gerais
    'brasil':     '#27ae60',   # verde — série referente ao Brasil
    'destaque':   '#e74c3c',   # vermelho — marcadores de atenção ou outliers
    'neutro':     '#95a5a6',   # cinza — elementos secundários ou referência neutra
}

## 2. Carregamento e limpeza dos dados

### 1.1 Atlas Brasil — IDHM histórico (1991–2010)

In [ ]:
# Lê apenas as 17 primeiras colunas do Excel (range(17) = colunas 0 a 16),
# evitando importar colunas vazias ou auxiliares que costumam aparecer
# à direita em planilhas institucionais do Atlas Brasil / PNUD.
df_idh_raw = pd.read_excel(data_xlsx, usecols=range(17))

In [ ]:
# Remove linhas em que a primeira coluna (Município) está vazia.
# Essas linhas correspondem a cabeçalhos repetidos, totalizadores ou
# linhas em branco intercaladas no layout original da planilha.
df_idh_raw = df_idh_raw.dropna(subset=[df_idh_raw.columns[0]])

# Renomeando colunas
# Substitui os cabeçalhos originais (frequentemente longos ou com acentos
# inconsistentes) por nomes padronizados, sem espaços e com sufixo de ano.
# A convenção <Indicador>_<Ano> facilita seleções por padrão com filter()
# ou str.contains() mais adiante.
df_idh_raw.columns = [
    'Municipio',                                        # nome do município / agregado
    'Pop_0a1_1991', 'Pop_0a1_2000', 'Pop_0a1_2010',    # proporção de extremamente pobres (0 a 1 SM)
    'PEA_2000', 'PEA_2010',                             # População Economicamente Ativa
    'IDHM_2000', 'IDHM_2010',                           # IDHM composto
    'IDHM_Renda_1991', 'IDHM_Renda_2000', 'IDHM_Renda_2010',  # dimensão Renda
    'IDHM_Long_1991',  'IDHM_Long_2000',  'IDHM_Long_2010',   # dimensão Longevidade
    'IDHM_Educ_1991',  'IDHM_Educ_2000',  'IDHM_Educ_2010',   # dimensão Educação
]

# Separar Brasil, MG e municípios
# O arquivo original mistura três níveis na mesma coluna 'Municipio':
# agregados nacionais, estaduais e registros municipais identificados por "(MG)".
# Os três DataFrames abaixo isolam cada nível para análises comparativas.

# Linha de referência nacional (agregado Brasil)
df_brasil = df_idh_raw[df_idh_raw['Municipio'] == 'Brasil'].copy()
# Linha de referência estadual (agregado Minas Gerais)
df_mg     = df_idh_raw[df_idh_raw['Municipio'] == 'Minas Gerais'].copy()
# Registros municipais — identificados pelo sufixo "(MG)" no nome.
# na=False evita erro caso alguma célula seja NaN após o dropna anterior.
df_idh    = df_idh_raw[df_idh_raw['Municipio'].str.contains(r'\(MG\)', na=False)].copy()

# Limpar nome do município
# Remove o sufixo " (MG)" do nome para deixar apenas o nome limpo
# (ex.: "Uberlândia (MG)" → "Uberlândia"), facilitando exibição em gráficos
# e merges futuros com outras bases que não usam esse sufixo.
df_idh['Municipio'] = df_idh['Municipio'].str.replace(r' \(MG\)', '', regex=True).str.strip()

# Verificação
print(f'IDH carregado: {len(df_idh)} municípios de MG')
display(df_idh[df_idh['Municipio'] == 'Uberlândia'][['Municipio','IDHM_2000','IDHM_2010']])

IDH carregado: 853 municípios de MG


,Municipio,IDHM_2000,IDHM_2010
826,Uberlândia,0.702,0.789


### 1.2 IMRS 2020 — Indicadores multidimensionais (FJP)